In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
import numpy as np
import gseapy as gp

In [ ]:
homer_path = ''
env = ''

In [ ]:
os.system(env+'python ./metilene3/metilene3.py \
    -i ./data/GSE121721_glioma.input.tsv \
    -o ./GBM \
    -t 16 \
    -n 4 \
    -plot True -anno hg19\
')

In [ ]:
os.system('cp ./GBM/DMRs-unsupervised.tsv ../SourceData/Fig.5b.txt')

In [ ]:
renameG = {
    'G0':'A',
    'G1':'B',
    'G2':'C',
    'G3':'D',
    'G4':'E',
    'G5':'F',
}

gbm_s = pd.read_table('./GBM/DMRs.tsv')
gbm_s['hypomethylated'] = gbm_s['Hypo-groups']
gbm_s['intermediate'] = gbm_s['Int-groups']
gbm_s['hypermethylated'] = gbm_s['Hyper-groups']
for i in renameG.keys():
    gbm_s['hypomethylated'] = gbm_s['hypomethylated'].str.replace(i,renameG[i])
    gbm_s['intermediate'] = gbm_s['intermediate'].str.replace(i,renameG[i])
    gbm_s['hypermethylated'] = gbm_s['hypermethylated'].str.replace(i,renameG[i])
gbm_s['mode'] = 'supervised'

# gbm = pd.concat([gbm_u, gbm_s]).sort_values(['mode','chr','start'])
gbm = pd.concat([gbm_s]).sort_values(['length','p-kwt'], ascending=[0,1])
gbm = gbm['chr	start	stop	meandiffabs	length	p-kwt	hypomethylated	intermediate	hypermethylated'.split('\t')]
gbm.to_csv('./figures/ST3.tsv',sep='\t', index=False)
gbm

In [ ]:
dmrmean_m_rename = pd.read_table('./GBM/heatmap.tsv', index_col=0)
sinfo = dmrmean_m_rename[[]]
sinfo['group'] = [i.split(' ')[0] for i in dmrmean_m_rename.index]
sinfo['subtype'] = [i.split('=')[1].split('_')[0] for i in dmrmean_m_rename.index]
sinfo.index = [i.split('_')[-1] for i in sinfo.index]
sinfo

In [ ]:
colors = dmrmean_m_rename[[]]

colors['group'] = [i.split(' ')[0] for i in dmrmean_m_rename.index]
clsc = {
    'G0':sns.color_palette("Set2")[3],
    'G1':sns.color_palette("Set2")[1],
    'G2':sns.color_palette("Set2")[2],
    'G3':sns.color_palette("Set2")[0],
}
colors['subtype'] = [i.split('=')[1].split('_')[0] for i in dmrmean_m_rename.index]
colors['groupc'] = colors['group'].map(clsc)
colors['subtypec'] = colors['subtype'].map({'normal':'green','wt':'red','R132H':'orange'})
colors.head()

In [ ]:
cm = sns.clustermap(dmrmean_m_rename,\
        row_colors=[colors['groupc'],\
                    colors['subtype'].map({'normal':'white','wt':'white','R132H':'white'}),\
                    colors['subtypec'],\
                    colors['subtype'].map({'normal':'white','wt':'white','R132H':'white'}),\
                    ],\
        col_cluster=False,row_cluster=False,\
        cmap='Spectral_r', dendrogram_ratio=0.000001, xticklabels=False, yticklabels=False, \
        method='ward', cbar_pos=None, vmax=1, vmin=0, center=0.5, colors_ratio=0.03)

plt.savefig('./figures/5b-r.pdf', bbox_inches='tight')

In [ ]:
from Bio import Phylo

tree = Phylo.read("./GBM/DMTree.nwk", "newick")

def change_labels(clade):
    if clade.name:
        clade.name = clade.name.split('_')[-1]+'-'.join(['' for i in range(20)])
    for subclade in clade.clades:
        change_labels(subclade)

change_labels(tree.root)

cmap = colors['groupc'].to_dict()
for i in colors.index:
    cmap[i.split('_')[-1]+'-'.join(['' for i in range(20)])] = cmap[i]
f,a = plt.subplots(figsize=[6,10])
Phylo.draw(tree, axes=a, do_show=False, label_colors=cmap,show_confidence=True)
plt.xscale('symlog')
plt.xlim([-0.1,1e5/2])
a.spines['top'].set_visible(False)
a.spines['left'].set_visible(False)
a.spines['right'].set_visible(False)
a.yaxis.set_visible(False)
plt.savefig('./figures/5b.pdf', bbox_inches='tight')

In [ ]:
met = pd.read_table('./data/GSE121721_glioma.input.tsv', na_values=['.'])
met.columns = [i.split('=')[-1] for i in met.columns]
met

In [ ]:
cpgstd = met.drop(columns=['chrom','end']).T.std()
cpgstd

In [ ]:
udmrs = pd.read_table('./GBM/DMRs-unsupervised.tsv',skiprows=2)
dmtncpg = udmrs.loc[(udmrs['meandiffabs']>0.5)&(udmrs['#Hyper']>=2)&(udmrs['#Hypo']>=2)]['length'].sum()
dmtncpg

In [ ]:
udmr4pca = []
udmrs.loc[(udmrs['meandiffabs']>0.5)&(udmrs['#Hyper']>=2)&(udmrs['#Hypo']>=2)]['mean'].apply(lambda x:udmr4pca.append(x.split('|')))
udmr4pca = pd.DataFrame(udmr4pca).astype(float).T
udmr4pca.index = met.columns[2:]
udmr4pca

In [ ]:
f,ax = plt.subplots(1,4,figsize=[12,3])

clsc = {
    'G0':sns.color_palette("Set2")[3],
    'G1':sns.color_palette("Set2")[1],
    'G2':sns.color_palette("Set2")[2],
    'G3':sns.color_palette("Set2")[0],
}

import numpy as np
from sklearn.decomposition import PCA

sd_gbm_pcas = []
models = ['all CpGs','top 1% CpGs','eq. #CpGs','unsupervised DMRs']

for ii, tmp in enumerate([
    np.array(met.drop(columns=['chrom','end']).T),
    np.array(met.drop(columns=['chrom','end']).loc[cpgstd>cpgstd.quantile(0.99)].T),
    np.array(met.drop(columns=['chrom','end']).loc[cpgstd>=list(cpgstd.sort_values())[-dmtncpg]].T),
    udmr4pca
]):
    pca = PCA(n_components=2)
    print(tmp.shape)
    X = pd.DataFrame(pca.fit_transform(tmp))
    X['model'] = models[ii]
    sd_gbm_pcas.append(X)
    print(pca.explained_variance_ratio_)
    X.index = [i.split('_')[-1] for i in met.columns[2:]]
    X['grp'] = X.index.map(sinfo['group'])
    X['subtype'] = X.index.map(sinfo['subtype'])
    X['batch'] = X.index.str.contains('_B')
    a = sns.scatterplot(x=X[0],y=X[1],hue=X['grp'],s=50, style=X['subtype'], markers={'wt':'s','R132H':'o','normal':'X'},palette=clsc, ax=ax[ii], legend=None)
    a.spines['top'].set_visible(False)
    # a.spines['left'].set_visible(False)
    a.spines['right'].set_visible(False)
    # a.yaxis.set_visible(False)
    a.set_title('n='+str(tmp.shape[1]))
    a.set_ylabel('PC2('+str(pca.explained_variance_ratio_[1]*100)[:5]+'%)')
    a.set_xlabel('PC1('+str(pca.explained_variance_ratio_[0]*100)[:5]+'%)')
    a.set_box_aspect(1)
    a.set_xticks(np.linspace(*a.get_xlim(), 3))
    a.set_yticks(np.linspace(*a.get_ylim(), 3))

f.tight_layout()
plt.savefig('./figures/ED6a.pdf', bbox_inches='tight')

In [ ]:
pd.concat(sd_gbm_pcas)[[0,1,'model']].to_csv('../SourceData/Fig.ED6a.txt',sep='\t')
pd.concat(sd_gbm_pcas)

In [ ]:
dmrs = pd.read_table('./GBM/DMRs.tsv')
dmrs

RNA

In [ ]:
bed = pd.read_table('./data/geo/GSE121720_RAW/gencode.v19.annotation.onlygenes.bed', header=None)
bed.index = bed[0]+'.'+bed[1].astype(str)+'.'+bed[2].astype(str)

raw = pd.read_table('./data/geo/raw.genes.npz')
raw.index = raw["#'chr'"]+'.'+raw["'start'"].astype(str)+'.'+raw["'end'"].astype(str)

raw = pd.DataFrame(raw.drop(columns=["#'chr'","'start'","'end'"]))
raw.columns = [i.split('_')[1] for i in raw.columns]
raw.index = raw.index.map(bed[3].to_dict())
raw.index.name = 'ensgene'
raw = raw.loc[~raw.index.duplicated()]

normal = {
    'frontal':'NBr1',
    'occipital':'NBr2',
    'parietal':'NBr3',
    'temporal':'NBr4'
}
newidx = list(raw.columns)
for i in range(len(newidx)):
    if newidx[i] in normal.keys():
        newidx[i] = normal[newidx[i]]
raw.columns = newidx
raw = raw.loc[raw.index[(raw.T>1).mean()>0.5]]
raw

In [ ]:
tpm = pd.read_table('./data/geo/GSE121720_RNAseq_expression_matrix_TPMs.txt.gz').T
tpm = (tpm+0.1).applymap(np.log10)
normal = {
    'frontal':'NBr1',
    'occipital':'NBr2',
    'parietal':'NBr3',
    'temporal':'NBr4'
}
newidx = list(tpm.index)
for i in range(len(newidx)):
    if newidx[i] in normal.keys():
        newidx[i] = normal[newidx[i]]
tpm.index = newidx
tpm = tpm.loc[raw.columns]
tpm

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

pcatmp = pca.fit_transform(np.array(tpm[tpm.columns[(tpm-tpm.mean()).std()>(tpm-tpm.mean()).std().quantile(0.8)]]))
print(pca.explained_variance_ratio_)
pcatmp = pd.DataFrame(pcatmp)
pcatmp.index = tpm.index
pcatmp

In [ ]:
pca_sinfo = sinfo.copy()
pca_sinfo.index = [i.split('_')[-1] for i in pca_sinfo.index]
pcatmp['grp'] = pcatmp.index.map(pca_sinfo['group'])
pcatmp['subtype'] = pcatmp.index.map(pca_sinfo['subtype'])
pcatmp

In [ ]:
pcatmp[[0,1]].to_csv('../SourceData/Fig.ED6b.txt',sep='\t')

In [ ]:
import seaborn as sns

clsc = {
    'G0':sns.color_palette("Set2")[3],
    'G1':sns.color_palette("Set2")[1],
    'G2':sns.color_palette("Set2")[2],
    'G3':sns.color_palette("Set2")[0],
}

plt.subplots(figsize=[3,3])
sns.scatterplot(x=pcatmp[0],y=pcatmp[1],hue=pcatmp['grp'],\
                s=50, style=pcatmp['subtype'], markers={'wt':'s','R132H':'o','normal':'X'}, palette=clsc, legend=None)
plt.title('n='+str(tpm[tpm.columns[(tpm-tpm.mean()).std()>(tpm-tpm.mean()).std().quantile(0.8)]].shape[1]))
plt.xlabel('PC1('+str(pca.explained_variance_ratio_[0]*100)[:7]+'%)')
plt.ylabel('PC2('+str(pca.explained_variance_ratio_[1]*100)[:7]+'%)')
plt.savefig('./figures/ED6b.pdf')

In [ ]:
def cfun(x,i):
    tocolor = {'1':'#9dc2a9','2':'#e7e6e6','3':'#e4c198'}
    if x:
        x = x.split('|')
        return tocolor[x[i]]
    else:
        return 'white'

def plotGene(gene, fl=3000, fr=3000):
    ngroups = 4
    
    plt.subplots(figsize=[1.6,2.5])
    cmap2 = colors[['group','groupc']].groupby('group').first()['groupc'].to_dict()
    ax = sns.swarmplot(x=tpm.index.map(sinfo['subtype']), y=tpm[gene], hue=tpm.index.map(sinfo['group']), palette=cmap2, s=3)
    ax.get_legend().set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ab = bed.loc[bed[3]==gene].index[0].split('.')

In [ ]:
tpm[['ENSG00000148773.8','ENSG00000041982.10']].to_csv('../SourceData/Fig.5d.txt',sep='\t')
tpm[['ENSG00000148773.8','ENSG00000041982.10']]

In [ ]:
plotGene('ENSG00000148773.8')
plt.savefig('./figures/5d.pdf')

In [ ]:
plotGene('ENSG00000041982.10')
plt.savefig('./figures/5d-r.pdf')

In [ ]:
def rundeseq(Y0,Y1):
    i = ''.join(Y1)+'vs'+''.join(Y0)
    selected = sinfo.loc[sinfo['group'].isin(Y0+Y1)]
    selected['y'] = 1*(selected['group'].isin(Y0))
    
    raw[selected.index].astype(int).to_csv(\
        './GBM/deseq2_exp_'+i+'.csv')
    selected[['y']].to_csv(\
        './GBM/deseq2_y_'+i+'.csv')
    os.system('Rscript ./deseq2.r \
            ./GBM/deseq2_exp_'+i+'.csv \
            ./GBM/deseq2_y_'+i+'.csv \
            ./GBM/deseq2_out_'+i+'.csv')

In [ ]:
gene_names = {}
with open('./data/geo/Homo_sapiens.GRCh38.111.gtf', 'r') as gtf_file:
    for line in gtf_file:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        if len(fields) < 9:
            continue
        attributes = fields[8]
        attr_dict = {}
        if attributes.find('gene_id')>-1 and attributes.find('gene_name')>-1 :
            gene_names[attributes.split('gene_id "')[1].split('"')[0]] = \
                attributes.split('gene_name "')[1].split('"')[0]

In [ ]:
Y1 = ['G2']
Y0 = ['G1']


rundeseq(Y0,Y1)

selected = sinfo.loc[sinfo['group'].isin(Y0+Y1)]
selected['y'] = 1*(selected['group'].isin(Y1))

i = ''.join(Y1)+'vs'+''.join(Y0)

deseq2 = pd.read_csv('./GBM/deseq2_out_'+i+'.csv', index_col=0)
deseq2['ENSG'] = deseq2.index 
deseq2.index = [j.split('.')[0] for j in deseq2.index]
deseq2.index = deseq2.index.map(gene_names)
deseq2 = deseq2.sort_values('pvalue')
deseq2 = deseq2.loc[(~deseq2.index.isna())&(~deseq2.index.duplicated())].dropna()

degsea = gp.prerank(rnk=deseq2['log2FoldChange'],
                     gene_sets='./data/geo/h.all.v2023.2.Hs.symbols.gmt',
                     threads=10, outdir=None, seed=1, 
                    )

degsea.res2d['absNES'] = degsea.res2d['NES'].apply(abs)
degsea.res2d.sort_values('absNES', ascending=False).head(10)

In [ ]:
pd.read_csv('./GBM/deseq2_out_G2vsG1.csv', index_col=0).sort_index().to_csv('./figures/ST4.tsv',sep='\t', index=False)
pd.read_csv('./GBM/deseq2_out_G2vsG1.csv', index_col=0).sort_index()

In [ ]:
i = 'P0|2|3|0'
gs = sorted(set(dmrs.loc[(dmrs['DMTree'].str.contains(i.replace('|','\|')))&\
                            (dmrs['meandiffabs']>0.1)]['SYMBOL'].dropna()))
print(i,len(gs))
dmgsea = gp.enrichr(gene_list=gs,
                 gene_sets='./data/geo/h.all.v2023.2.Hs.symbols.gmt',
                 outdir=None,
                )

In [ ]:
dgsea = pd.merge(dmgsea.res2d, degsea.res2d, on='Term', how='outer').dropna()

dgsea['combine_pvalues'] = dgsea.apply(lambda x:st.combine_pvalues([x['P-value'],x['NOM p-val']]).pvalue,axis=1)
dgsea['combine_pvalues'] += dgsea.loc[dgsea['combine_pvalues']>0]['combine_pvalues'].min()*0.1

plt.subplots(figsize=[8,4])
ax = sns.scatterplot(x=dgsea['Odds Ratio'].apply(np.log2), y=dgsea['NES'],\
                     s=30-30*dgsea['combine_pvalues'].apply(np.log10), color='gray')

ax = sns.scatterplot(x=dgsea.loc[dgsea['combine_pvalues']<0.01]['Odds Ratio'].apply(np.log2), y=dgsea.loc[dgsea['combine_pvalues']<0.01]['NES'],\
                     s=30-30*dgsea.loc[dgsea['combine_pvalues']<0.01]['combine_pvalues'].apply(np.log10), color='orange')


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.yticks([0,1,2,2.2])
plt.xticks([-1.5,0,1.5])

plt.title(st.spearmanr(dgsea['Odds Ratio'], dgsea['NES'], nan_policy='omit'))
dgsea.loc[dgsea['combine_pvalues']<0.01].sort_values(['NES'])

plt.savefig('./figures/5c.pdf')

In [ ]:
dgsea = pd.merge(dmgsea.res2d, degsea.res2d, on='Term', how='outer').dropna()
dgsea['combine_pvalues'] = dgsea.apply(lambda x:st.combine_pvalues([x['P-value'],x['NOM p-val']]).pvalue,axis=1)
dgsea.to_csv('./figures/ST5.tsv',sep='\t', index=False)
dgsea.to_csv('../SourceData/Fig.5c.txt',sep='\t', index=False)
dgsea

In [ ]:
dmrs = pd.read_table('./GBM/DMRs.tsv').sort_values(['chr','start','stop'])
dmrs['strand'] = '+'
dmrs['stop_1'] = dmrs['stop']+1

path = './GBM/motif/'
os.system('mkdir '+path)
for i in ['P0|2|3|0']:
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv(path+i.replace('|','_')+'.bed', sep='\t', header=False)
    dmrs.loc[(dmrs['meandiffabs']>=0.5)&(~dmrs['DMTree'].fillna('').str.contains(i.replace('|','\|')+','))][['chr','start','stop_1','strand']].\
        to_csv(path+i.replace('|','_')+'.anti.bed', sep='\t', header=False)
    os.system(env+homer_path+'findMotifsGenome.pl '+path+i.replace('|','_')+'.bed'+\
                  ' hg19'+\
                  ' '+path+i.replace('|','_')+' -bits -size 250  -bg '+path+i.replace('|','_')+'.anti.bed ')

In [ ]:
os.system('cp ./GBM/motif/P0_2_3_0.bed ../SourceData/Fig.ED6c-part1.txt')
os.system('cp ./GBM/motif/P0_2_3_0.anti.bed ../SourceData/Fig.ED6c-part2.txt')

In [ ]:
os.system('cp ./GBM/motif/P0_2_3_0/homerResults.html ./figures/ED6c.html')